# Evaluate Your Model with Tau-Bench (Agentic Benchmark)

## What is Tau-Bench?

[Tau-Bench](https://github.com/sierra-research/tau2-bench) (Tau2) is an agentic benchmark that evaluates language models on realistic, multi-turn tool-use customer service scenarios. The model must use tools, follow policies, and complete tasks across multiple conversation turns.

### Domains

| Domain | Description | Scoring |
|--------|-------------|---------|
| `tau2_airline` | Airline customer service (booking changes, cancellations) | Tool calls + goal assessment via grading model |
| `tau2_retail` | Retail customer service (orders, returns) | Tool calls + information relay verification |
| `tau2_telecom` | Telecom customer service (plans, billing) | Tool calls + end state comparison |

### How It Works

Tau-bench uses **two models**:

| Role | Purpose |
|------|---------|
| **Agent** (primary model) | The model being evaluated — handles tool calls, takes actions |
| **User** (simulator) | Simulates a customer giving instructions to the agent |

## Two Approaches

| Approach | How it runs | Best for |
|----------|------------|----------|
| **Local SDK** | `inspect eval` CLI on your machine | Fast iteration, development |
| **Container Job** | SageMaker Training Job | Scalable, hands-off, production |

## Supported Inference Providers

| Provider | Model Source |
|----------|-------------|
| **Bedrock** | [Custom Model Deployment](https://docs.aws.amazon.com/bedrock/latest/userguide/deploy-custom-model-on-demand.html) (on-demand) |
| **SageMaker** | [SageMaker Endpoint](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-deploy-models.html) (OpenAI-compatible). See [Evaluate Models on SageMaker](https://docs.aws.amazon.com/nova/latest/userguide/nova-eval-on-sagemaker-inference.html). |

## Prerequisites

1. **AWS credentials configured** — via `aws configure`, environment variables, or IAM role
2. **A model to evaluate** — either a Bedrock custom model or a running SageMaker endpoint
3. **Python 3.10+**

### Token Usage Warning

Tau-bench uses a **high amount of tokens** (~60M for airline domain on a full run). Use `LIMIT` and `MESSAGE_LIMIT` parameters to control costs during testing.

**Time to complete:** ~10-30 min (limited) | hours (full) | **Cost:** Inference costs + `ml.m5.large` (~ash.13/hr) for Approach 2

## Configuration

In [ ]:
# =============================================================================
# UPDATE THESE VALUES
# =============================================================================

# Inference provider: "bedrock" or "sagemaker"
INFERENCE_PROVIDER = "sagemaker"

# --- Bedrock Configuration ---
# ARN of your custom model (from fine-tuning, distillation, or import)
CUSTOM_MODEL_ARN = "arn:aws:bedrock:us-east-1:123456789012:custom-model/amazon.nova-2-lite-v1:0:256k/your-model-id"
DEPLOYMENT_NAME = "my-nova-lite2-tau-bench"  # Name for the Custom Model Deployment

# --- SageMaker Configuration ---
ENDPOINT_NAME = "SAGEMAKER_ENDPOINT_NAME"  # SageMaker endpoint name

# --- Common Configuration ---
REGION = "us-east-1"

# Evaluation parameters
DOMAIN = "tau2_airline"  # Options: tau2_airline, tau2_retail, tau2_telecom
LIMIT = 5               # Number of samples (set to None for full eval)
MESSAGE_LIMIT = 10      # Max messages per sample (100 for full eval)

# User simulator model (simulates the customer)
USER_MODEL = "bedrock/us.anthropic.claude-sonnet-4-6"

# =============================================================================
print(f"Provider: {INFERENCE_PROVIDER}")
print(f"Region: {REGION}")
print(f"Domain: {DOMAIN}")
print(f"Limit: {LIMIT or "Full"}")
print(f"Message limit: {MESSAGE_LIMIT}")
if INFERENCE_PROVIDER == "bedrock":
    print(f"Custom model: {CUSTOM_MODEL_ARN}")
else:
    print(f"Endpoint: {ENDPOINT_NAME}")

### Verify Your Model

In [ ]:
import boto3, json, time
from datetime import datetime

if INFERENCE_PROVIDER == "bedrock":
    bedrock_client = boto3.client("bedrock", region_name=REGION)
    bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

    # Check for existing active deployment
    response = bedrock_client.list_custom_model_deployments(modelArnEquals=CUSTOM_MODEL_ARN)
    active = [d for d in response.get("customModelDeploymentSummaries", []) if d["status"] == "Active"]

    if active:
        DEPLOYMENT_ARN = active[0]["customModelDeploymentArn"]
        print(f"✓ Existing active deployment: {DEPLOYMENT_ARN}")
    else:
        print("No active deployment found. Creating one...")
        resp = bedrock_client.create_custom_model_deployment(
            modelDeploymentName=f"{DEPLOYMENT_NAME}-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
            modelArn=CUSTOM_MODEL_ARN,
        )
        DEPLOYMENT_ARN = resp["customModelDeploymentArn"]
        print(f"✓ Created: {DEPLOYMENT_ARN}")
        while True:
            status = bedrock_client.get_custom_model_deployment(
                customModelDeploymentIdentifier=DEPLOYMENT_ARN
            )["status"]
            if status == "Active":
                print("✓ Deployment is active!")
                break
            elif status == "Failed":
                raise Exception("Deployment failed")
            print(f"  Status: {status}...", end="\r")
            time.sleep(30)

    MODEL_ID = f"bedrock/{DEPLOYMENT_ARN}"

elif INFERENCE_PROVIDER == "sagemaker":
    sm_client = boto3.client("sagemaker", region_name=REGION)
    try:
        resp = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
        if resp["EndpointStatus"] == "InService":
            print(f"Endpoint '{ENDPOINT_NAME}' is InService")
        else:
            print(f"Endpoint status: {resp["EndpointStatus"]}. Wait for InService.")
    except Exception as e:
        print(f"✗ Endpoint not found: {e}")
    MODEL_ID = f"sagemaker/{ENDPOINT_NAME}"

print(f"\nModel ID for eval: {MODEL_ID}")

### Quick Inference Test (Tool Calling)

In [ ]:
if INFERENCE_PROVIDER == "bedrock":
    response = bedrock_runtime.converse(
        modelId=DEPLOYMENT_ARN,
        messages=[{"role": "user", "content": [{"text": "What's the weather in Seattle?"}]}],
        toolConfig={"tools": [{
            "toolSpec": {
                "name": "get_weather",
                "description": "Get current weather for a location",
                "inputSchema": {"json": {
                    "type": "object",
                    "properties": {"location": {"type": "string", "description": "City name"}},
                    "required": ["location"]
                }}
            }
        }]},
        inferenceConfig={"maxTokens": 200, "temperature": 0.7},
    )
    output = response["output"]["message"]
    for block in output["content"]:
        if "text" in block:
            print(f"Text: {block['text']}")
        if "toolUse" in block:
            print(f"✓ Tool call: {block['toolUse']['name']}")
            print(f"  Input: {json.dumps(block['toolUse']['input'], indent=2)}")

elif INFERENCE_PROVIDER == "sagemaker":
    sm_runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    payload = {
        "messages": [{"role": "user", "content": "What's the weather in Seattle?"}],
        "tools": [{
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get current weather for a location",
                "parameters": {
                    "type": "object",
                    "properties": {"location": {"type": "string", "description": "City name"}},
                    "required": ["location"]
                }
            }
        }],
        "max_tokens": 200,
        "temperature": 0.7,
    }
    response = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(payload),
    )
    result = json.loads(response["Body"].read())
    choice = result["choices"][0]["message"]
    if choice.get("content"):
        print(f"Text: {choice['content']}")
    if choice.get("tool_calls"):
        for tc in choice["tool_calls"]:
            print(f"✓ Tool call: {tc['function']['name']}")
            print(f"  Input: {tc['function']['arguments']}")


---

## Approach 1: Local SDK (Quick Evaluation)

Run evaluations directly on your machine using the `inspect eval` CLI. Best for fast iteration and development.

In [ ]:
!pip install inspect-ai inspect-evals aioboto3 openai --quiet

### Run Evaluation

In [ ]:
import subprocess, os

os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["AWS_REGION"] = REGION

# Build command
cmd = f"inspect eval inspect_evals/{DOMAIN} --model "{MODEL_ID}""

# Provider-specific args
if INFERENCE_PROVIDER == "sagemaker":
    cmd += f" -M region_name={REGION} -M read_timeout=600"
elif INFERENCE_PROVIDER == "bedrock":
    cmd += f" -M region_name={REGION} -M read_timeout=600"

# User simulator
cmd += f" --model-role "user={USER_MODEL}""

# Eval parameters
if LIMIT:
    cmd += f" --limit {LIMIT}"
cmd += f" -T message_limit={MESSAGE_LIMIT}"
cmd += " --temperature 0.0 --max-tokens 8192"
cmd += " --max-connections 3 --max-retries 10 --display plain"

print(f"Running:
{cmd}
")
subprocess.run(cmd, shell=True)

In [ ]:
!inspect view

---

## Approach 2: Container Job (Scalable Evaluation)

Run the same evaluation as a SageMaker Training Job. The container:
1. Downloads your config and benchmarks from S3
2. Installs benchmark dependencies
3. Sends requests to your model for each sample
4. Publishes results to S3 incrementally

No GPU needed — the `ml.m5.large` orchestrator only coordinates the eval.

In [ ]:
!pip install "boto3" "sagemaker" pyyaml --quiet

### Setup: IAM Role and S3 Bucket

In [ ]:
import json, os, time, yaml
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, Compute
from sagemaker.core.shapes.shapes import StoppingCondition, OutputDataConfig

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
S3_BUCKET = f"inspect-tau-bench-{ACCOUNT_ID}"
ROLE_NAME = "InspectLensEvalRole"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/sagemaker-inspect-ai:latest"

# Create IAM role
iam = boto3.client("iam")
trust_policy = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "sagemaker.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
try:
    iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"✓ Created role: {ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"✓ Role exists: {ROLE_NAME}")
policies = ["arn:aws:iam::aws:policy/AmazonSageMakerFullAccess", "arn:aws:iam::aws:policy/AmazonS3FullAccess"]
for arn in policies:
    iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
time.sleep(10)

# Create S3 bucket
s3 = boto3.client("s3", region_name=REGION)
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(Bucket=S3_BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"✓ Created bucket: {S3_BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"✓ Bucket exists: {S3_BUCKET}")

### Create and Upload Benchmarks

In [ ]:
os.makedirs("benchmarks", exist_ok=True)

domains = {
    "tau2_airline": "from inspect_evals.tau2 import tau2_airline as _t\nfrom inspect_ai import task\n\n@task\ndef tau2_airline(**kwargs):\n    return _t(**kwargs)\n",
    "tau2_retail": "from inspect_evals.tau2 import tau2_retail as _t\nfrom inspect_ai import task\n\n@task\ndef tau2_retail(**kwargs):\n    return _t(**kwargs)\n",
    "tau2_telecom": "from inspect_evals.tau2 import tau2_telecom as _t\nfrom inspect_ai import task\n\n@task\ndef tau2_telecom(**kwargs):\n    return _t(**kwargs)\n",
}
for name, code in domains.items():
    with open(f"benchmarks/{name}.py", "w") as f:
        f.write(code)
with open("benchmarks/requirements.txt", "w") as f:
    f.write("inspect-evals\n")

for root, _, files in os.walk("benchmarks"):
    for fn in files:
        s3.upload_file(os.path.join(root, fn), S3_BUCKET, os.path.join(root, fn))
print("✓ Benchmarks uploaded")

### Write Eval Config

In [ ]:
if INFERENCE_PROVIDER == "bedrock":
    provider_config = {"bedrock_model": {"model_id": DEPLOYMENT_ARN, "region": REGION}}
else:
    provider_config = {"sagemaker_endpoint": {"endpoint_name": ENDPOINT_NAME, "region": REGION}}

task_config = {"name": DOMAIN, "task_args": {"message_limit": MESSAGE_LIMIT}}
if LIMIT:
    task_config["limit"] = LIMIT

config = {
    "inference_provider": provider_config,
    "benchmarks": {"s3_path": f"s3://{S3_BUCKET}/benchmarks/", "tasks": [task_config]},
    "eval": {
        "max_connections": 3,
        "max_retries": 10,
        "timeout": 600,
        "decoding": {"temperature": 0.0, "max_tokens": 8192},
    },
    "output": {"s3_path": f"s3://{S3_BUCKET}/eval-results/"},
}

os.makedirs("config", exist_ok=True)
with open("config/inspect_config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
s3.upload_file("config/inspect_config.yaml", S3_BUCKET, "config/inspect_config.yaml")
print(f"✓ Config uploaded ({INFERENCE_PROVIDER} mode)")
print("---")
print(yaml.dump(config, default_flow_style=False))

### Submit and Monitor

In [ ]:
sagemaker_client = boto3.client("sagemaker", region_name=REGION)

trainer = ModelTrainer(
    training_image=IMAGE_URI, role=ROLE_ARN,
    compute=Compute(instance_type="ml.m5.large", instance_count=1, volume_size_in_gb=30),
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{S3_BUCKET}/output/"),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=86400),
    base_job_name="inspect-tau-bench",
)
trainer.train(
    input_data_config=[InputData(channel_name="config", data_source=f"s3://{S3_BUCKET}/config/")],
    wait=False,
)
job_name = trainer._latest_training_job.training_job_name
print(f"✓ Job: {job_name}")
print(f"  https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{job_name}")

# Monitor
while True:
    resp = sagemaker_client.describe_training_job(TrainingJobName=job_name)
    status = resp["TrainingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        print(f"\n✓ {status}")
        if status == "Failed":
            print(f"  Reason: {resp.get("FailureReason")}")
        break
    print(f"  {status} / {resp.get("SecondaryStatus", "")}...", end="\r")
    time.sleep(15)

### View Results

In [ ]:
os.makedirs("results", exist_ok=True)
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f"eval-results/{job_name}/eval_results/")
if "Contents" in resp:
    for obj in resp["Contents"]:
        s3.download_file(S3_BUCKET, obj["Key"], os.path.join("results", os.path.basename(obj["Key"])))
        print(f"  Downloaded: {os.path.basename(obj["Key"])}")
    with open("results/_status.json") as f:
        st = json.load(f)
    print(f"\nStatus: {st["status"]}")
    print(f"Passed: {st["passed_tasks"]}")
    print(f"Failed: {st["failed_tasks"]}")
else:
    print("No results found. Check job status above.")

---

## Cleanup

Remove resources created during evaluation.

- **Bedrock:** Deletes the Custom Model Deployment (your custom model is preserved)
- **SageMaker:** Your endpoint is NOT deleted (delete separately if no longer needed)
- **Container Job:** Deletes the S3 bucket and IAM role created for the job

In [ ]:
# Delete Bedrock Custom Model Deployment (if created)
if INFERENCE_PROVIDER == "bedrock":
    try:
        bedrock_client.delete_custom_model_deployment(
            customModelDeploymentIdentifier=DEPLOYMENT_ARN
        )
        print(f"✓ Deleted Bedrock deployment: {DEPLOYMENT_ARN}")
    except Exception as e:
        print(f"⚠ Could not delete deployment: {e}")

# Delete S3 and IAM from container approach (if used)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=S3_BUCKET):
        if "Contents" in page:
            s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": [{"Key": o["Key"]} for o in page["Contents"]]})
    s3.delete_bucket(Bucket=S3_BUCKET)
    print(f"✓ Deleted bucket: {S3_BUCKET}")
    for arn in policies:
        iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
    iam.delete_role(RoleName=ROLE_NAME)
    print(f"✓ Deleted role: {ROLE_NAME}")
except Exception as e:
    print(f"⚠ Container cleanup skipped: {e}")

# Optional: Delete SageMaker endpoint (uncomment if no longer needed)
# if INFERENCE_PROVIDER == "sagemaker":
#     sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
#     print(f"✓ Deleted endpoint: {ENDPOINT_NAME}")

import shutil
for d in ["benchmarks", "config", "results"]:
    shutil.rmtree(d, ignore_errors=True)
print("✓ Local files cleaned up")

---

## Reference

### Key Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `--max-connections` | 10 | Parallel requests. Scale with instance count (e.g., 10 instances x 25 = 250). |
| `--max-retries` | 3 | Retry attempts for failed requests. Use 50-100 for large evaluations. |
| `-M region_name` | us-east-1 | AWS region where your model is deployed. |
| `-M read_timeout` | 600 | Request timeout in seconds. |
| `-T message_limit` | - | Max conversation turns per sample. Use 100 for full eval. |


### Full Evaluation

To run the full benchmark, set:


### Troubleshooting

| Issue | Cause | Fix |
|-------|-------|-----|
| Endpoint throttling or timeouts | Too many parallel requests | Reduce `--max-connections`, increase `--max-retries` |
| Authentication errors | Missing IAM permissions | Verify credentials include `sagemaker:InvokeEndpoint` or `bedrock:InvokeModel` |
| Model not found (Bedrock) | Deployment not active | Check deployment status with `get_custom_model_deployment` |
| Endpoint not found (SageMaker) | Wrong region or name | Verify with `aws sagemaker list-endpoints --region REGION` |
| Tool calling failures | Model doesn't support tools | Verify model was fine-tuned with tool calling data |
| Many errors (E) in results | Infrastructure issue | Check CloudWatch metrics for capacity issues |

### Resources

- [Inspect AI Documentation](https://inspect.ai-safety-institute.org.uk/)
- [Inspect Evals Repository](https://github.com/UKGovernmentBEIS/inspect_evals)
- [Tau2 Paper](https://arxiv.org/abs/2506.07982)
- [Tau2 Leaderboard](https://taubench.com/#leaderboard)
- [Evaluate Models on SageMaker Inference](https://docs.aws.amazon.com/nova/latest/userguide/nova-eval-on-sagemaker-inference.html)
- [Deploy Custom Model on Bedrock (On-Demand)](https://docs.aws.amazon.com/bedrock/latest/userguide/deploy-custom-model-on-demand.html)
- [Deploy SageMaker Real-Time Endpoints](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-deploy-models.html)